# Chapter 19 — Let the Machine Touch Something

**Companion to Applied AI**

Question: How do we know the file the agent claims to have written is the file on disk?

By the end of this notebook you will have:

- separated proposal, authorization, action, and observation
- let a process write only inside a temp sandbox
- caught lying, partial, and wrong-target adapters against observed bytes

## What this notebook demonstrates
`decided ≠ requested ≠ performed ≠ observed`: the worker's report is a claim; a runtime re-read is the evidence. All writes go to a temp directory.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
import os, tempfile, hashlib, json

seed: 42


## 1. Proposal and authorization come before any touch

In [2]:
sandbox = tempfile.mkdtemp(prefix="applied-ai-19-")
proposal = {"action": "write_file", "path": os.path.join(sandbox, "note.txt"),
            "bytes": b"canary 10%"}
grant = {"allows": ["write_file"], "confined_to": sandbox}
assert proposal["path"].startswith(grant["confined_to"])
print("proposal:", {k: (v if k != "bytes" else v.decode()) for k, v in proposal.items()})
assert os.path.isdir(sandbox)

proposal: {'action': 'write_file', 'path': 'C:\\Users\\ernan\\AppData\\Local\\Temp\\applied-ai-19-n2wec07q\\note.txt', 'bytes': 'canary 10%'}


## 2. Four adapters: honest, lying, partial, wrong-target

In [3]:
def run_adapter(mode: str):
    target = proposal["path"] if mode != "wrong-target" else os.path.join(sandbox, "elsewhere.txt")
    payload = proposal["bytes"] if mode in ("honest", "lying") else proposal["bytes"][:6]
    if mode != "lying":
        with open(target, "wb") as f:
            f.write(payload)
    report = {"claimed": "wrote full payload to " + proposal["path"]}
    # the runtime re-read: only disk state counts, never the worker's report
    observed = open(proposal["path"], "rb").read() if os.path.exists(proposal["path"]) else None
    ok = observed == proposal["bytes"]
    obs_hash = hashlib.sha256(observed).hexdigest()[:12] if observed is not None else None
    return {"mode": mode, "report": report["claimed"], "observed_hash": obs_hash, "accepted": ok}

for mode in ["honest", "lying", "partial", "wrong-target"]:
    if os.path.exists(proposal["path"]):
        os.remove(proposal["path"])
    if os.path.exists(os.path.join(sandbox, "elsewhere.txt")):
        os.remove(os.path.join(sandbox, "elsewhere.txt"))
    print(run_adapter(mode))

{'mode': 'honest', 'report': 'wrote full payload to C:\\Users\\ernan\\AppData\\Local\\Temp\\applied-ai-19-n2wec07q\\note.txt', 'observed_hash': '378c3055b7e7', 'accepted': True}
{'mode': 'lying', 'report': 'wrote full payload to C:\\Users\\ernan\\AppData\\Local\\Temp\\applied-ai-19-n2wec07q\\note.txt', 'observed_hash': None, 'accepted': False}
{'mode': 'partial', 'report': 'wrote full payload to C:\\Users\\ernan\\AppData\\Local\\Temp\\applied-ai-19-n2wec07q\\note.txt', 'observed_hash': 'e100fbce008c', 'accepted': False}
{'mode': 'wrong-target', 'report': 'wrote full payload to C:\\Users\\ernan\\AppData\\Local\\Temp\\applied-ai-19-n2wec07q\\note.txt', 'observed_hash': None, 'accepted': False}


## Interpretation
- Supports: generation and effect are separate events; only the runtime's re-read counts as observation.
- Does NOT support: a real sandbox (no permissions, no processes here).

## Try it yourself
1. Revoke the grant and show the action refused before any write.
2. Add a READ action and count provider calls for each mode.
3. Require `observed_hash` in the acceptance record.